# AI Gateway Auto Router with LangChain

Auto Router chooses a model from an explicit Workers AI candidate list for each chat request. This example uses `ChatCloudflareWorkersAI` through the REST API and shows the serving model and routing metadata on the returned `AIMessage`.

Run `make notebook_sync` and `make notebook_lab` from the repository root. The notebook environment installs this checkout of `langchain-cloudflare`. Set `CF_ACCOUNT_ID`, `CF_AI_API_TOKEN`, and `AI_GATEWAY` in the repo-root `.env`. Use a gateway in the same Cloudflare account.

[Cloudflare Auto Router documentation](https://developers.cloudflare.com/ai-gateway/features/auto-router/)

In [ ]:
import os
import uuid

from dotenv import load_dotenv
from langchain_cloudflare import ChatCloudflareWorkersAI
from langchain_core.messages import HumanMessage

load_dotenv(".env")
account_id = os.getenv("CF_ACCOUNT_ID")
api_token = os.getenv("CF_AI_API_TOKEN")
gateway_id = os.getenv("AI_GATEWAY")
missing = [
    name
    for name, value in {
        "CF_ACCOUNT_ID": account_id,
        "CF_AI_API_TOKEN": api_token,
        "AI_GATEWAY": gateway_id,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(
        f"Set these variables before running the notebook: {', '.join(missing)}"
    )

## Select from Workers AI candidates

`aig_allowed_models` replaces Auto Router's default pool. This library requires an explicit list of Workers AI model IDs (`@cf/...`) for `cloudflare/auto`. REST calls use the OpenAI-compatible endpoint.

In [ ]:
candidates = [
    "@cf/qwen/qwen3.8-27b",
    "@cf/deepseek-ai/deepseek-v4-flash-0731",
]
router = ChatCloudflareWorkersAI(
    model="cloudflare/auto",
    account_id=account_id,
    api_token=api_token,
    ai_gateway=gateway_id,
    endpoint_format="openai_compatible",
    aig_allowed_models=candidates,
)
reply = router.invoke("Say hello briefly.")
print(reply.content)
print(
    {
        "requested_model": reply.response_metadata.get("requested_model"),
        "serving_model": reply.response_metadata.get("model_name"),
        "ai_gateway": reply.response_metadata.get("ai_gateway"),
    }
)

## Keep a model within a conversation turn

Cloudflare's `cf-aig-session-id` groups requests in a conversation. `cf-aig-turn-id` identifies one user message and its follow-up requests. Reuse the session ID across the conversation and set a new turn ID for each new user turn. These differ from Workers AI's existing `session_id` option, which sends `x-session-affinity`.

[Session affinity documentation](https://developers.cloudflare.com/ai-gateway/features/auto-router/#session-affinity)

In [ ]:
conversation_id = f"notebook-{uuid.uuid4().hex}"
turn_id = f"{conversation_id}-turn-1"
turn_router = ChatCloudflareWorkersAI(
    model="cloudflare/auto",
    account_id=account_id,
    api_token=api_token,
    ai_gateway=gateway_id,
    endpoint_format="openai_compatible",
    aig_allowed_models=candidates,
    aig_session_id=conversation_id,
    aig_turn_id=turn_id,
)
turn_reply = turn_router.invoke("Give one short greeting.")
print(turn_reply.content)
print(turn_reply.response_metadata.get("ai_gateway"))

## Vision request

Keep only vision-capable models in the candidate list for image requests. The text-only candidate above cannot accept `image_url`; this example uses Qwen alone. The embedded PNG is a 16×16 red square, so no image file or download is needed.

In [ ]:
red_png_base64 = (
    "iVBORw0KGgoAAAANSUhEUgAAABAAAAAQCAIAAACQkWg2AAAAF0lEQVR4nGP4z8BA"
    "EiJN9aiGUQ1DSgMAkPn/Afnh+ngAAAAASUVORK5CYII="
)
vision_router = ChatCloudflareWorkersAI(
    model="cloudflare/auto",
    account_id=account_id,
    api_token=api_token,
    ai_gateway=gateway_id,
    endpoint_format="openai_compatible",
    aig_allowed_models=["@cf/qwen/qwen3.8-27b"],
)
image_message = HumanMessage(
    content=[
        {"type": "text", "text": "What is the dominant color in this image?"},
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{red_png_base64}"},
        },
    ]
)
vision_reply = vision_router.invoke([image_message])
print(vision_reply.content)
print(vision_reply.response_metadata.get("model_name"))

## Python Worker binding

A Jupyter kernel has no Worker `env.AI` binding. Inside a Python Worker, use the same model options with `binding=self.env.AI` and call `await router.ainvoke(...)`. The binding response exposes the selected model and routing reason/IDs in the same `response_metadata` keys.

```python
router = ChatCloudflareWorkersAI(
    model="cloudflare/auto",
    binding=self.env.AI,
    ai_gateway=gateway_id,
    aig_allowed_models=["@cf/qwen/qwen3.8-27b"],
)
reply = await router.ainvoke("Say hello briefly.")
print(reply.response_metadata)
```